# MNIST CNN: Inductive Biases

In [ ]:
! pip3 install -q flax jax optax torchvision

In [ ]:
import flax.linen as nn
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
from scipy.ndimage import rotate

## 1. Load and Explore MNIST

MNIST: 70k grayscale images of handwritten digits (28x28 pixels). Classify each as 0-9.

In [ ]:
from torchvision.datasets import MNIST

train_data = MNIST(root="./data", train=True, download=True)
test_data = MNIST(root="./data", train=False, download=True)

X_train = train_data.data.numpy().astype(np.float32) / 255.0
y_train = train_data.targets.numpy()
X_test = test_data.data.numpy().astype(np.float32) / 255.0
y_test = test_data.targets.numpy()

# Add channel dim: [N, 28, 28] -> [N, 28, 28, 1]
X_train = X_train[..., None]
X_test = X_test[..., None]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Pixel range: [{X_train.min()}, {X_train.max()}]")

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i, :, :, 0], cmap="gray")
    ax.set_title(y_train[i], fontsize=10)
    ax.axis("off")
plt.suptitle("MNIST training examples")
plt.tight_layout()
plt.show()

## 2. Define the CNN

A small CNN with two conv layers, max pooling, and a classification head:

```
Input [28, 28, 1]
  -> Conv(16, 3x3) + ReLU + MaxPool(2x2)   [13, 13, 16]
  -> Conv(32, 3x3) + ReLU + MaxPool(2x2)   [5, 5, 32]
  -> Flatten                                 [800]
  -> Dense(64) + ReLU -> Dense(10)           [10]
```

In [ ]:
class CNN(nn.Module):
    @nn.compact
    def __call__(self, x):
        x = nn.relu(nn.Conv(16, (3, 3))(x))
        x = nn.max_pool(x, (2, 2), strides=(2, 2))
        x = nn.relu(nn.Conv(32, (3, 3))(x))
        x = nn.max_pool(x, (2, 2), strides=(2, 2))
        x = x.reshape(x.shape[0], -1)
        x = nn.relu(nn.Dense(64)(x))
        return nn.Dense(10)(x)


model = CNN()
optimizer = optax.adam(1e-3)

# Check parameter count
dummy_params = model.init(jax.random.PRNGKey(0), jnp.ones((1, 28, 28, 1)))
n_params = sum(x.size for x in jax.tree_util.tree_leaves(dummy_params))
print(f"Parameters: {n_params:,}")

In [ ]:
@jax.jit
def train_step(params, opt_state, X, y):
    def loss_fn(p):
        logits = model.apply(p, X)
        return optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, new_opt_state = optimizer.update(grads, opt_state, params)
    return optax.apply_updates(params, updates), new_opt_state, loss


def evaluate(params, X, y, batch_size=512):
    """Compute accuracy in batches."""
    correct = 0
    for i in range(0, len(X), batch_size):
        logits = model.apply(params, X[i : i + batch_size])
        correct += (logits.argmax(-1) == y[i : i + batch_size]).sum()
    return float(correct / len(X))

## 3. Train on Standard MNIST

We train for a fixed number of gradient steps and evaluate periodically.

In [ ]:
n_steps = 500
eval_every = 50
batch_size = 128

params = model.init(jax.random.PRNGKey(0), jnp.ones((1, 28, 28, 1)))
opt_state = optimizer.init(params)

steps_log, train_accs, test_accs = [], [], []

for step in range(1, n_steps + 1):
    idx = np.random.randint(0, len(X_train), batch_size)
    params, opt_state, loss = train_step(params, opt_state, X_train[idx], y_train[idx])

    if step % eval_every == 0:
        train_acc = evaluate(params, X_train, y_train)
        test_acc = evaluate(params, X_test, y_test)
        steps_log.append(step)
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        print(f"Step {step:4d} | Train: {train_acc:.1%} | Test: {test_acc:.1%}")

params_standard = params

In [ ]:
plt.plot(steps_log, train_accs, "o-", label="Train")
plt.plot(steps_log, test_accs, "o-", label="Test")
plt.xlabel("Step")
plt.ylabel("Accuracy")
plt.title("Training on standard MNIST")
plt.legend()
plt.ylim(0.9, 1.0)
plt.show()

## 4. The Rotation Test

Our CNN reaches ~99% on the standard test set. But what happens when we rotate the digits by 90, 180, or 270 degrees?

CNNs are **translation invariant** (weight sharing), but not rotation invariant.

In [ ]:
def rotate_images(X, rng=None):
    """Rotate each image by a random multiple of 90 degrees (90, 180, or 270)."""
    if rng is None:
        rng = np.random.default_rng()
    X_rot = np.zeros_like(X)
    angles = rng.choice([90, 180, 270], size=len(X))
    for i in range(len(X)):
        X_rot[i, :, :, 0] = rotate(X[i, :, :, 0], angles[i], reshape=False, order=1)
    return np.clip(X_rot, 0, 1), angles


# Generate rotated test set
X_test_rot, angles = rotate_images(X_test, rng=np.random.default_rng(42))

# Show clean vs rotated examples
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for i in range(8):
    axes[0, i].imshow(X_test[i, :, :, 0], cmap="gray")
    axes[0, i].set_title(f"label={y_test[i]}")
    axes[0, i].axis("off")
    axes[1, i].imshow(X_test_rot[i, :, :, 0], cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original", fontsize=10)
axes[1, 0].set_ylabel("Rotated", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
clean_acc = test_accs[-1]
rot_acc = evaluate(params_standard, X_test_rot, y_test)
print(f"Clean test accuracy:   {clean_acc:.1%}")
print(f"Rotated test accuracy: {rot_acc:.1%}")

## 5. Data Augmentation

The fix: augment the training data with random rotations so the network learns rotation-robust features.

We apply a random rotation to each image on the fly during training. The network never sees the same image twice.

In [ ]:
def augment_batch(X):
    """Apply random 90/180/270 degree rotation to a batch."""
    X_aug = np.zeros_like(X)
    for i in range(len(X)):
        angle = np.random.choice([0, 90, 180, 270])
        # angle = np.random.uniform(0, 360)  # Random angle between 0 and 360 degrees
        X_aug[i, :, :, 0] = rotate(X[i, :, :, 0], angle, reshape=False, order=1)
    return np.clip(X_aug, 0, 1)


# Fresh model, same architecture
params_aug = model.init(jax.random.PRNGKey(0), jnp.ones((1, 28, 28, 1)))
opt_state_aug = optimizer.init(params_aug)

steps_log_aug, train_accs_aug, test_accs_aug, rot_accs_aug = [], [], [], []

for step in range(1, n_steps + 1):
    idx = np.random.randint(0, len(X_train), batch_size)
    X_batch = augment_batch(X_train[idx])
    params_aug, opt_state_aug, loss = train_step(params_aug, opt_state_aug, X_batch, y_train[idx])

    if step % eval_every == 0:
        train_acc = evaluate(params_aug, X_train, y_train)
        test_acc = evaluate(params_aug, X_test, y_test)
        rot_acc_aug = evaluate(params_aug, X_test_rot, y_test)
        steps_log_aug.append(step)
        train_accs_aug.append(train_acc)
        test_accs_aug.append(test_acc)
        rot_accs_aug.append(rot_acc_aug)
        print(f"Step {step:4d} | Train: {train_acc:.1%} | Test: {test_acc:.1%} | Rotated: {rot_acc_aug:.1%}")

## 6. Compare

How does the augmented model compare to the standard model across different rotation angles?

In [ ]:
# Evaluate both models at each rotation angle
test_angles = [0, 90, 180, 270]
acc_standard = []
acc_augmented = []

for angle in test_angles:
    if angle == 0:
        X_rot = X_test
    else:
        # Rotate all images by exactly this angle
        X_rot = np.zeros_like(X_test)
        for i in range(len(X_test)):
            X_rot[i, :, :, 0] = rotate(X_test[i, :, :, 0], angle, reshape=False, order=1)
        X_rot = np.clip(X_rot, 0, 1)
    acc_standard.append(evaluate(params_standard, X_rot, y_test))
    acc_augmented.append(evaluate(params_aug, X_rot, y_test))
    print(f"{angle:3d}\u00b0  Standard: {acc_standard[-1]:.1%}  Augmented: {acc_augmented[-1]:.1%}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Left: accuracy per rotation angle
x = np.arange(len(test_angles))
width = 0.3
ax1.bar(x - width / 2, acc_standard, width, label="Standard")
ax1.bar(x + width / 2, acc_augmented, width, label="Augmented")
ax1.set_xlabel("Rotation angle")
ax1.set_ylabel("Accuracy")
ax1.set_xticks(x)
ax1.set_xticklabels([f"{a}\u00b0" for a in test_angles])
ax1.legend()
ax1.set_ylim(0, 1.05)
ax1.set_title("Accuracy by rotation angle")

# Right: training curves (rotated test accuracy over steps)
ax2.plot(
    steps_log,
    [evaluate(params_standard, X_test_rot, y_test)] * len(steps_log),
    "--",
    color="C0",
    alpha=0.5,
    label="Standard (rotated)",
)
ax2.plot(steps_log, [test_accs[-1]] * len(steps_log), "-", color="C0", alpha=0.5, label="Standard (clean)")
ax2.plot(steps_log_aug, rot_accs_aug, "s-", color="C1", label="Augmented (rotated)")
ax2.plot(steps_log_aug, test_accs_aug, "o-", color="C1", alpha=0.5, label="Augmented (clean)")
ax2.set_xlabel("Step")
ax2.set_ylabel("Accuracy")
ax2.set_title("Training progress")
ax2.legend(fontsize=8)
ax2.set_ylim(0.3, 1.02)

plt.tight_layout()
plt.show()